In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from scipy.optimize import linear_sum_assignment
from torchvision.ops import generalized_box_iou, box_convert

ModuleNotFoundError: No module named 'timm'

In [ ]:
# ----------------------------------------------------------------------------
# Backbone: small ResNet-style CNN, output stride 16
# ----------------------------------------------------------------------------
class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.skip = None
        if stride != 1 or c_in != c_out:
            self.skip = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False),
                                      nn.BatchNorm2d(c_out))
 
    def forward(self, x):
        idt = x if self.skip is None else self.skip(x)
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.relu(x + idt)

In [ ]:
class SmallBackbone(nn.Module):
    """stem(/2) -> stage1(/4) -> stage2(/8) -> stage3(/16)"""
    def __init__(self, in_ch=1, widths=(32, 64, 128, 256)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, widths[0], 3, 2, 1, bias=False),
            nn.BatchNorm2d(widths[0]), nn.ReLU(inplace=True))
        layers, c = [], widths[0]
        for i, w in enumerate(widths[1:]):
            layers += [BasicBlock(c, w, stride=2), BasicBlock(w, w)]
            c = w
        self.body = nn.Sequential(*layers)
        self.out_channels = c
 
    def forward(self, x):
        return self.body(self.stem(x))

In [ ]:
class MobileNetV4Backbone(nn.Module):
    """MobileNetV4 from timm, returning only the stride-16 feature map.
    Variants: mobilenetv4_conv_small (default), mobilenetv4_conv_medium, mobilenetv4_hybrid_medium, ...
    in_ch != 3 is handled by timm, which adapts the pretrained first conv to the new channel count."""
    def __init__(self, name='mobilenetv4_conv_small', pretrained=True, in_ch=3, stride=16):
        super().__init__()
        probe = timm.create_model(name, pretrained=False, features_only=True)
        idx = probe.feature_info.reduction().index(stride)
        self.body = timm.create_model(name, pretrained=pretrained, features_only=True,
                                      out_indices=(idx,), in_chans=in_ch)
        self.out_channels = self.body.feature_info.channels()[-1]
 
    def forward(self, x):
        return self.body(x)[-1]

In [ ]:
# ----------------------------------------------------------------------------
# 2D sine positional encoding (as in DETR)
# ----------------------------------------------------------------------------
def sine_pos_embed(h, w, dim, device, temperature=10000.0):
    assert dim % 4 == 0
    y, x = torch.meshgrid(torch.arange(h, device=device, dtype=torch.float32),
                          torch.arange(w, device=device, dtype=torch.float32), indexing="ij")
    y = (y + 0.5) / h * 2 * math.pi
    x = (x + 0.5) / w * 2 * math.pi
    d = dim // 4
    omega = 1.0 / temperature ** (torch.arange(d, device=device, dtype=torch.float32) / d)
    y = y.flatten()[:, None] * omega[None]
    x = x.flatten()[:, None] * omega[None]
    return torch.cat([x.sin(), x.cos(), y.sin(), y.cos()], dim=1)  # (h*w, dim)

In [ ]:
# ----------------------------------------------------------------------------
# Transformer (pre-norm). Positional encodings are added to Q/K, not V, as in DETR.
# ----------------------------------------------------------------------------
class EncoderLayer(nn.Module):
    def __init__(self, d, heads, ffn, drop):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, ffn), nn.GELU(), nn.Dropout(drop), nn.Linear(ffn, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)
 
    def forward(self, src, pos):
        q = k = self.n1(src) + pos
        src = src + self.drop(self.attn(q, k, self.n1(src))[0])
        return src + self.drop(self.ff(self.n2(src)))
 
 
class DecoderLayer(nn.Module):
    def __init__(self, d, heads, ffn, drop):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, ffn), nn.GELU(), nn.Dropout(drop), nn.Linear(ffn, d))
        self.n1, self.n2, self.n3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)
        self.drop = nn.Dropout(drop)
 
    def forward(self, tgt, memory, pos, query_pos):
        t = self.n1(tgt)
        tgt = tgt + self.drop(self.self_attn(t + query_pos, t + query_pos, t)[0])
        t = self.n2(tgt)
        tgt = tgt + self.drop(self.cross_attn(t + query_pos, memory + pos, memory)[0])
        return tgt + self.drop(self.ff(self.n3(tgt)))
 
 
class MLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, n_layers):
        super().__init__()
        dims = [d_in] + [d_hidden] * (n_layers - 1) + [d_out]
        self.layers = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))
 
    def forward(self, x):
        for i, l in enumerate(self.layers):
            x = l(x) if i == len(self.layers) - 1 else F.relu(l(x))
        return x

In [ ]:
# ----------------------------------------------------------------------------
# GalaxyDETR
# ----------------------------------------------------------------------------
class GalaxyDETR(nn.Module):
    def __init__(self, num_classes, in_ch=3, d_model=128, heads=4, enc_layers=3,
                 dec_layers=3, ffn=512, num_queries=30, dropout=0.1,
                 backbone='mobilenetv4_conv_small', pretrained=True):
        super().__init__()
        # backbone='custom' uses the small from-scratch ResNet-style CNN instead
        self.backbone = (SmallBackbone(in_ch) if backbone == 'custom'
                         else MobileNetV4Backbone(backbone, pretrained, in_ch))
        self.proj = nn.Conv2d(self.backbone.out_channels, d_model, 1)
        self.encoder = nn.ModuleList(EncoderLayer(d_model, heads, ffn, dropout) for _ in range(enc_layers))
        self.enc_norm = nn.LayerNorm(d_model)
        self.decoder = nn.ModuleList(DecoderLayer(d_model, heads, ffn, dropout) for _ in range(dec_layers))
        self.dec_norm = nn.LayerNorm(d_model)
        self.query_embed = nn.Embedding(num_queries, d_model)
        self.class_head = nn.Linear(d_model, num_classes + 1)  # last index = "no galaxy"
        self.box_head = MLP(d_model, d_model, 4, 3)
        self.d_model = d_model
 
    def forward(self, x):
        feat = self.proj(self.backbone(x))               # (B, d, H, W)
        B, d, H, W = feat.shape
        src = feat.flatten(2).transpose(1, 2)            # (B, HW, d)
        pos = sine_pos_embed(H, W, d, x.device)[None]    # (1, HW, d)
 
        for layer in self.encoder:
            src = layer(src, pos)
        memory = self.enc_norm(src)
 
        query_pos = self.query_embed.weight[None].expand(B, -1, -1)
        tgt = torch.zeros_like(query_pos)
        outs = []
        for layer in self.decoder:
            tgt = layer(tgt, memory, pos, query_pos)
            outs.append(self.dec_norm(tgt))
        hs = torch.stack(outs)                           # (L, B, Q, d)
 
        logits = self.class_head(hs)
        boxes = self.box_head(hs).sigmoid()
        out = {"pred_logits": logits[-1], "pred_boxes": boxes[-1]}
        out["aux"] = [{"pred_logits": l, "pred_boxes": b} for l, b in zip(logits[:-1], boxes[:-1])]
        return out

In [ ]:
# ----------------------------------------------------------------------------
# Hungarian matcher + set criterion
# ----------------------------------------------------------------------------
def cxcywh_to_xyxy(b):
    return box_convert(b, "cxcywh", "xyxy")
 
 
class HungarianMatcher(nn.Module):
    def __init__(self, w_class=2.0, w_bbox=5.0, w_giou=2.0):
        super().__init__()
        self.wc, self.wb, self.wg = w_class, w_bbox, w_giou
 
    @torch.no_grad()
    def forward(self, outputs, targets):
        B, Q = outputs["pred_logits"].shape[:2]
        prob = outputs["pred_logits"].flatten(0, 1).softmax(-1)
        pbox = outputs["pred_boxes"].flatten(0, 1)
        tcls = torch.cat([t["labels"] for t in targets])
        tbox = torch.cat([t["boxes"] for t in targets])
        if len(tcls) == 0:
            e = torch.empty(0, dtype=torch.long)
            return [(e, e) for _ in targets]
        C = (self.wb * torch.cdist(pbox, tbox, p=1)
             - self.wc * prob[:, tcls]
             - self.wg * generalized_box_iou(cxcywh_to_xyxy(pbox), cxcywh_to_xyxy(tbox)))
        C = C.view(B, Q, -1).cpu()
        sizes = [len(t["boxes"]) for t in targets]
        idx = [linear_sum_assignment(c[i].numpy()) for i, c in enumerate(C.split(sizes, -1))]
        return [(torch.as_tensor(i, dtype=torch.long), torch.as_tensor(j, dtype=torch.long)) for i, j in idx]
 
 
class SetCriterion(nn.Module):
    def __init__(self, num_classes, matcher, eos_coef=0.1, w_ce=1.0, w_bbox=5.0, w_giou=2.0):
        super().__init__()
        self.nc, self.matcher = num_classes, matcher
        self.w = {"ce": w_ce, "bbox": w_bbox, "giou": w_giou}
        weight = torch.ones(num_classes + 1)
        weight[-1] = eos_coef
        self.register_buffer("ce_weight", weight)
 
    def single(self, out, targets, n_boxes):
        idx = self.matcher(out, targets)
        b_idx = torch.cat([torch.full_like(s, i) for i, (s, _) in enumerate(idx)])
        s_idx = torch.cat([s for s, _ in idx])
        tgt_cls = torch.full(out["pred_logits"].shape[:2], self.nc, dtype=torch.long,
                             device=out["pred_logits"].device)
        tgt_cls[b_idx, s_idx] = torch.cat([t["labels"][j] for t, (_, j) in zip(targets, idx)])
        l_ce = F.cross_entropy(out["pred_logits"].transpose(1, 2), tgt_cls, self.ce_weight)
 
        src = out["pred_boxes"][b_idx, s_idx]
        tgt = torch.cat([t["boxes"][j] for t, (_, j) in zip(targets, idx)])
        l_bbox = F.l1_loss(src, tgt, reduction="sum") / n_boxes
        l_giou = (1 - torch.diag(generalized_box_iou(cxcywh_to_xyxy(src), cxcywh_to_xyxy(tgt)))).sum() / n_boxes
        return self.w["ce"] * l_ce + self.w["bbox"] * l_bbox + self.w["giou"] * l_giou, \
            {"ce": l_ce.item(), "bbox": l_bbox.item(), "giou": l_giou.item()}
 
    def forward(self, outputs, targets):
        n = max(sum(len(t["labels"]) for t in targets), 1)
        total, logs = self.single(outputs, targets, n)
        for aux in outputs["aux"]:
            total = total + self.single(aux, targets, n)[0]
        return total, logs
 
 
@torch.no_grad()
def postprocess(outputs, threshold=0.5):
    """Return per-image list of dicts with scores, labels, boxes (cxcywh, normalized)."""
    prob = outputs["pred_logits"].softmax(-1)[..., :-1]
    scores, labels = prob.max(-1)
    res = []
    for s, l, b in zip(scores, labels, outputs["pred_boxes"]):
        keep = s > threshold
        res.append({"scores": s[keep], "labels": l[keep], "boxes": b[keep]})
    return res

In [ ]:
# ----------------------------------------------------------------------------
# Data: the WHOLE Galaxy10 DECaLS dataset from the .h5 file, split 80% train / 20% validation.
# Same loading pattern as vgg_train_and_val.ipynb: keep the raw uint8 images (~3.5 GB) in RAM and
# preprocess each image as the DataLoader asks for it.
#
# Galaxy10 has a class label per image but NO bounding boxes. Every cutout is centered on its
# galaxy, so we derive one box per image from the central light blob (pseudo-label):
#   threshold at background median + 3*sigma (MAD estimate) -> connected components ->
#   take the component containing / nearest the center -> pad its bounding box by 10%.
# ----------------------------------------------------------------------------
import os
import time
import h5py
import numpy as np
import torchvision
from PIL import Image
from scipy import ndimage
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
H5_PATH = os.path.join('..', 'data', 'Galaxy10_DECals.h5')
class_names = ['disturbed', 'merging', 'round_smooth', 'in_between_round_smooth', 'cigar_smooth',
               'barred_spiral', 'unbarred_tight_spiral', 'unbarred_loose_spiral',
               'edge_on_no_bulge', 'edge_on_with_bulge']  # label = index in this list (0-9)
num_classes = len(class_names)
 
# Geometry steps are split from the tensor steps so the box is computed on exactly the
# pixels the model sees (256 -> center-crop 224). Combined, this is the notebook's pipeline.
geometry = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
])
to_tensor = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
def galaxy_box(img, nsigma=3.0, pad=0.10, fallback=0.3):
    """Box (cx, cy, w, h), normalized, around the central galaxy of an (H, W, 3) uint8 image."""
    gray = np.asarray(img, dtype=np.float32).mean(axis=2)
    H, W = gray.shape
    med = np.median(gray)
    sigma = 1.4826 * np.median(np.abs(gray - med)) + 1e-6
    mask = ndimage.binary_opening(gray > med + nsigma * sigma, iterations=1)
    comps, n = ndimage.label(mask)
    if n == 0:
        return [0.5, 0.5, fallback, fallback]
    c = comps[H // 2, W // 2]
    if c == 0:  # center pixel not above threshold: use the component nearest the center
        ys, xs = np.nonzero(comps)
        k = np.argmin((ys - H / 2) ** 2 + (xs - W / 2) ** 2)
        c = comps[ys[k], xs[k]]
    ys, xs = np.nonzero(comps == c)
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    w, h = (x1 - x0) * (1 + 2 * pad), (y1 - y0) * (1 + 2 * pad)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    return [cx / W, cy / H, min(w / W, 1.0), min(h / H, 1.0)]

In [ ]:
def load_galaxy10(h5_path=H5_PATH):
    with h5py.File(h5_path, 'r') as f:
        images = f['images'][:]                  # (17736, 256, 256, 3) uint8
        labels = f['ans'][:].astype(np.int64)    # (17736,)
    print('Loaded', images.shape, labels.shape)
    # Stratified so train and validation have the same class mix
    train_idx, val_idx = train_test_split(np.arange(len(labels)), test_size=0.20,
                                          stratify=labels, random_state=42)
    for name, Y in [('train', labels[train_idx]), ('validation', labels[val_idx])]:
        print(name, Y.shape, 'class counts:', np.bincount(Y, minlength=num_classes))
    return images, labels, train_idx, val_idx

In [ ]:
class GalaxyDataset(Dataset):
    """Returns (image through the preprocessing, target), image shape (3, 224, 224) float32,
    target = {'boxes': (1, 4) cxcywh normalized, 'labels': (1,)}."""
    def __init__(self, images, labels, indices):
        self.images, self.labels, self.indices = images, labels, indices
 
    def __len__(self):
        return len(self.indices)
 
    def __getitem__(self, i):
        j = self.indices[i]
        img = geometry(Image.fromarray(self.images[j]).convert('RGB'))
        target = {'boxes': torch.tensor([galaxy_box(img)], dtype=torch.float32),
                  'labels': torch.tensor([int(self.labels[j])])}
        return to_tensor(img), target

In [ ]:
def get_device():
    device = torch.device('cpu')
    if torch.cuda.is_available():
        device = torch.device('cuda')  # rose laptops and the servers
    elif torch.backends.mps.is_available():
        device = torch.device('mps')   # mac
    print('Using device:', device)
    return device

In [ ]:
# ----------------------------------------------------------------------------
# Training / evaluation (same structure as the notebook: best-val-loss checkpoint, then
# accuracy + confusion matrix on the validation set)
# ----------------------------------------------------------------------------
def to_device(tgts, device):
    return [{k: v.to(device) for k, v in t.items()} for t in tgts]
 
 
def train_model(model, criterion, optimizer, train_loader, val_loader, device, num_epochs, checkpoint_path):
    best_validation_loss, best_validation_epoch = float('inf'), -1
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_epochs)
    training_start = time.time()
    for epoch in range(num_epochs):
        model.train()
        epoch_loss, num_batches = 0.0, 0
        for images, targets in train_loader:
            loss, _ = criterion(model(images.to(device)), to_device(targets, device))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1)
            optimizer.step()
            epoch_loss += loss.item()
            num_batches += 1
        sched.step()
        avg_loss = epoch_loss / num_batches
 
        model.eval()
        with torch.no_grad():
            val_total, n_val = 0.0, 0
            for images, targets in val_loader:
                loss, _ = criterion(model(images.to(device)), to_device(targets, device))
                val_total += loss.item() * len(targets)
                n_val += len(targets)
            validation_loss = val_total / n_val
 
        best_marker = ''
        if validation_loss < best_validation_loss:
            best_validation_loss, best_validation_epoch = validation_loss, epoch
            torch.save(model.state_dict(), checkpoint_path)
            best_marker = ' *'
        print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {avg_loss:.5f}, Val Loss: {validation_loss:.5f}{best_marker}',
              flush=True)
    print(f'Average time per epoch: {(time.time() - training_start) / num_epochs:.2f}s')
    print(f'Best validation loss: {best_validation_loss:.5f} (epoch {best_validation_epoch})')
 
 
def evaluate_model(model, val_loader, device, checkpoint_path):
    """Image-level accuracy: the class of the most confident non-'no galaxy' query,
    so the number is directly comparable to the VGG classifier's accuracy."""
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    preds, trues, ious = [], [], []
    with torch.no_grad():
        for images, targets in val_loader:
            out = model(images.to(device))
            prob = out['pred_logits'].softmax(-1)[..., :-1]       # (B, Q, C)
            score, cls = prob.max(-1)                              # (B, Q)
            best_q = score.argmax(-1)                              # (B,)
            b = torch.arange(len(best_q))
            preds.append(cls[b, best_q].cpu())
            box = out['pred_boxes'][b, best_q].cpu()
            gt = torch.cat([t['boxes'] for t in targets])
            ious.append(torch.diag(generalized_box_iou(cxcywh_to_xyxy(box), cxcywh_to_xyxy(gt))))
            trues.append(torch.cat([t['labels'] for t in targets]))
    preds, trues = torch.cat(preds).numpy(), torch.cat(trues).numpy()
    print(f'Validation accuracy: {(preds == trues).mean():.4f}')
    print(f'Mean GIoU of top box vs. pseudo-label box: {torch.cat(ious).mean():.4f}')
    confusion = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(trues, preds):
        confusion[t, p] += 1
    print('Confusion matrix (rows = true, cols = predicted):', class_names)
    print(confusion)
 
 
if __name__ == '__main__':
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument('--h5', default=H5_PATH)
    ap.add_argument('--epochs', type=int, default=50)
    ap.add_argument('--batch-size', type=int, default=32)
    ap.add_argument('--workers', type=int, default=2)
    ap.add_argument('--checkpoint', default='galaxy_detr.pth')
    ap.add_argument('--backbone', default='mobilenetv4_conv_small',
                    help="any timm mobilenetv4_* name, or 'custom' for the from-scratch CNN")
    ap.add_argument('--no-pretrained', action='store_true', help='train the backbone from scratch')
    args = ap.parse_args()
 
    torch.manual_seed(42)
    device = get_device()
    images, labels, train_idx, val_idx = load_galaxy10(args.h5)
    train_loader = DataLoader(GalaxyDataset(images, labels, train_idx), batch_size=args.batch_size,
                              shuffle=True, num_workers=args.workers, collate_fn=collate)
    val_loader = DataLoader(GalaxyDataset(images, labels, val_idx), batch_size=args.batch_size,
                            num_workers=args.workers, collate_fn=collate)
 
    # One galaxy per Galaxy10 cutout, so 10 queries is plenty (vs. 30 for full star fields)
    model = GalaxyDETR(num_classes=num_classes, in_ch=3, num_queries=10,
                       backbone=args.backbone, pretrained=not args.no_pretrained).to(device)
    print(f'Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')
    criterion = SetCriterion(num_classes, HungarianMatcher()).to(device)
    # DETR trick: lower LR on the (pretrained) backbone so its ImageNet features aren't wrecked early
    optimizer = torch.optim.AdamW([
        {'params': [p for n, p in model.named_parameters() if n.startswith('backbone')], 'lr': 1e-4},
        {'params': [p for n, p in model.named_parameters() if not n.startswith('backbone')], 'lr': 3e-4},
    ], weight_decay=1e-4)
 
    train_model(model, criterion, optimizer, train_loader, val_loader, device, args.epochs, args.checkpoint)
    evaluate_model(model, val_loader, device, args.checkpoint)